<a href="https://colab.research.google.com/github/pradeep-84/pinnacle-tasks/blob/main/Resume_praser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import pdfplumber
import spacy

# Load natural language processing model
nlp = spacy.load("en_core_web_sm")

# Define a predefined list of skills for keyword matching
SKILL_BANK = ["Python", "Java", "SQL", "React", "Machine Learning", "Data Analysis", "Project Management"]

def extract_text_from_pdf(pdf_path):
    """Extracts raw text from a PDF file."""
    with pdfplumber.open(pdf_path) as pdf:
        text = ""
        for page in pdf.pages:
            text += page.extract_text() + "\n"
    return text

def parse_resume(raw_text):
    """Parses raw text to extract structured entities."""
    parsed_data = {
        "name": None,
        "email": None,
        "phone": None,
        "skills": []
    }

    # 1. Extract Email using Regular Expressions
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    email_match = re.search(email_pattern, raw_text)
    if email_match:
        parsed_data["email"] = email_match.group(0)

    # 2. Extract Phone Number using Regular Expressions
    phone_pattern = r'\b(?:\+?\d{1,3}[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b'
    phone_match = re.search(phone_pattern, raw_text)
    if phone_match:
        parsed_data["phone"] = phone_match.group(0)

    # 3. Extract Name using NLP Named Entity Recognition (NER)
    doc = nlp(raw_text)
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            # Usually, the first person entity mentioned at the top is the applicant
            parsed_data["name"] = ent.text.strip()
            break

    # 4. Extract Skills using Keyword Matching
    for skill in SKILL_BANK:
        # Case-insensitive matching with word boundaries
        if re.search(r'\b' + re.escape(skill) + r'\b', raw_text, re.IGNORECASE):
            parsed_data["skills"].append(skill)

    return parsed_data

# --- Example Usage ---
# text = extract_text_from_pdf("my_resume.pdf")
sample_resume_text = """
Pradeep reddy
Artificial intelligence engineer
Email: gurupradeepreddy84@gmail.com
phone:9999999999

Experience:
Built scalable web apps using Python and React. Managed SQL databases.
"""

result = parse_resume(sample_resume_text)
print(result)

{'name': None, 'email': 'gurupradeepreddy84@gmail.com', 'phone': '9999999999', 'skills': ['Python', 'SQL', 'React']}
